# YOLOv8 Arrow Detection Training (COCO Export)

This notebook converts the COCO export to YOLO format, creates a train/val split, then trains a YOLOv8 model.

In [4]:
# Optional (run if dependencies are not installed yet)
# !pip install -r requirements.txt

In [8]:
from pathlib import Path
import json
import random
import shutil
import yaml
from ultralytics import YOLO

In [9]:
# Paths + split config
COCO_JSON = Path('game arrows.coco/train/_annotations.coco.json')
IMAGE_DIR = COCO_JSON.parent
OUTPUT_ROOT = Path('arrow_dataset')
TRAIN_RATIO = 0.8
SEED = 42

assert COCO_JSON.exists(), f'Missing COCO JSON: {COCO_JSON.resolve()}'
print('COCO JSON:', COCO_JSON.resolve())

COCO JSON: C:\Users\user\Documents\arrow_clicker\game arrows.coco\train\_annotations.coco.json


In [11]:
# Convert COCO -> YOLO + split into train/val
with COCO_JSON.open('r', encoding='utf-8') as f:
    coco = json.load(f)

# We want class order: left, right, up, down (direction-sensitive)
name_order = ['left', 'right', 'up', 'down']
name_to_index = {name: idx for idx, name in enumerate(name_order)}
id_to_name = {c['id']: c['name'] for c in coco['categories']}
id_to_index = {cid: name_to_index[name] for cid, name in id_to_name.items() if name in name_to_index}

images = coco['images']
random.seed(SEED)
random.shuffle(images)
split_idx = int(len(images) * TRAIN_RATIO)
train_images = images[:split_idx]
val_images = images[split_idx:]

annotations_by_image = {}
for ann in coco['annotations']:
    annotations_by_image.setdefault(ann['image_id'], []).append(ann)

def prepare_split(split_name, split_images):
    img_out = OUTPUT_ROOT / 'images' / split_name
    lbl_out = OUTPUT_ROOT / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img in split_images:
        src = IMAGE_DIR / img['file_name']
        dst = img_out / img['file_name']
        if not dst.exists():
            shutil.copy2(src, dst)

        label_lines = []
        for ann in annotations_by_image.get(img['id'], []):
            if ann['category_id'] not in id_to_index:
                continue
            class_id = id_to_index[ann['category_id']]
            x, y, w, h = map(float, ann['bbox'])
            x_center = (x + w / 2) / img['width']
            y_center = (y + h / 2) / img['height']
            w_norm = w / img['width']
            h_norm = h / img['height']
            label_lines.append('{:d} {:.6f} {:.6f} {:.6f} {:.6f}'.format(class_id, x_center, y_center, w_norm, h_norm))

        label_path = lbl_out / f'{Path(img["file_name"]).stem}.txt'
        label_path.write_text('\n'.join(label_lines))

prepare_split('train', train_images)
prepare_split('val', val_images)

DATA_YAML = OUTPUT_ROOT / 'data.yaml'
data_yaml = {
    'path': str(OUTPUT_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': name_order,
    'nc': len(name_order)
}
DATA_YAML.write_text(yaml.safe_dump(data_yaml, sort_keys=False))

print('Created:', DATA_YAML.resolve())
print('Train images:', len(train_images), 'Val images:', len(val_images))

Created: C:\Users\user\Documents\arrow_clicker\arrow_dataset\data.yaml
Train images: 44 Val images: 12


In [12]:
# Training settings tuned for small, direction-sensitive dataset
MODEL_WEIGHTS = 'yolov8n.pt'
EPOCHS = 120
IMGSZ = 640
BATCH = 8
DEVICE = 0  # set to 'cpu' if no GPU
PROJECT = 'runs'
RUN_NAME = 'arrow_yolov8n_coco_small'

In [13]:
# Train YOLOv8 (augmentations avoid flips to preserve direction labels)
model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    patience=30,
    lr0=0.002,
    lrf=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    mosaic=0.8,
    mixup=0.1,
    fliplr=0.0,
    flipud=0.0
)
train_results

Ultralytics 8.4.19  Python-3.12.10 torch-2.10.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
# Validate using best checkpoint
best_model_path = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(best_model_path)
val_metrics = best_model.val(data=str(DATA_YAML))
val_metrics

In [ ]:
# Run predictions on validation images
pred_results = best_model.predict(
    source=str(OUTPUT_ROOT / 'images' / 'val'),
    conf=0.25,
    save=True,
    project=PROJECT,
    name=f'{RUN_NAME}_predict'
)
print('Saved predictions to:', Path(PROJECT) / f'{RUN_NAME}_predict')

## Notes
- Best weights are saved at: `runs/arrow_yolov8n_coco_small/weights/best.pt`.
- Augmentations disable flips to preserve left/right/up/down semantics.
- If training overfits quickly, reduce `EPOCHS` or increase `patience`/augmentations.